# 模型部署与服务化

> 机制都讲完了：调度、PagedAttention、Prefix Cache、PD 分离、量化、投机解码——它们不是知识点，是真实推理引擎里的代码。这一章把它们跑起来。
>
> 目标很具体：用 vLLM 把一个 Hugging Face 模型变成 OpenAI-compatible API，用 SDK 调用它，从流式响应里亲手测出 TTFT 和 TPOT，再换 SGLang 对比一遍。做完这一章，Part 3 从「读过」变成「跑过」。
>
> 本章五块内容：
>
> 1. **启动**：vLLM / SGLang 的最小部署与常见启动参数。
> 2. **调用**：健康检查、OpenAI SDK、流式输出。
> 3. **测量**：从 chunk 时间戳算出 TTFT / TPOT，做一次 8 并发小压测。
> 4. **排障**：OOM、TTFT 高、TPOT 高的排查顺序。
> 5. **对比**：怎么公平地比较两个引擎。

## 1. 从 Checkpoint 到 Serving Engine

`model.generate()` 适合开发验证，但离「服务」还差一整层。中间这层就是前面几本讲的推理引擎，它把散装的 checkpoint 组装成一个常驻进程：

```text
Hugging Face checkpoint
      ↓
Tokenizer / Chat Template
      ↓
vLLM / SGLang / llama.cpp / TensorRT-LLM   <- 推理引擎
      ↓
scheduler + KV cache + kernels             <- 前几章的机制都在这层
      ↓
OpenAI-compatible HTTP API
      ↓
client / gateway / application
```

启动一个引擎时，它在幕后依次做的是：找到并加载权重、按配置分配 KV Cache 显存、初始化 kernel、起调度循环、最后挂上 HTTP 服务。对照这张图，接下来动手把每一层跑通。

## 2. vLLM 的最小启动

截至 2026 年，vLLM 官方 Quickstart 推荐用 `uv` 管理环境：

```bash
uv venv --python 3.12 --seed
source .venv/bin/activate
uv pip install vllm --torch-backend=auto

vllm serve Qwen/Qwen3-0.6B --host 0.0.0.0 --port 8000
```

安装方式随硬件（CUDA / ROCm / 其他）各有差别，以官方文档为准；部署教程不该把某条 wheel 命令当成永远不变的知识——**真正稳定的是 serve 背后的数据流和参数含义**。先把环境摸清楚：

In [ ]:
# 部署前先看环境：有没有 NVIDIA GPU、有没有装 vllm
import importlib.util
import sys

try:
    import torch
    has_cuda = torch.cuda.is_available()
    gpu_name = torch.cuda.get_device_name(0) if has_cuda else ""
except Exception:
    has_cuda, gpu_name = False, ""

has_vllm = importlib.util.find_spec("vllm") is not None

print(f"Python     : {sys.version.split()[0]}")
print(f"NVIDIA GPU : {('有 - ' + gpu_name) if has_cuda else '本机没有'}")
print(f"vllm 包     : {'已安装' if has_vllm else '未安装'}")
print()
if has_cuda:
    print("可以在本机直接跑下面的 vllm serve；它会占住 GPU，建议放在独立终端运行")
else:
    print("本机没有 NVIDIA GPU 时，下面的启动命令保留完整步骤，在有 GPU 的机器上执行；")
    print("后面几节的客户端 cell 连得上任何一台 vLLM 服务器都能跑")

### 2.1 启动一个量化模型

22 一章的量化实战里亲手做过 GPTQ / AWQ / FP8 / GGUF，这里看它们怎么变成服务。三种典型启动：

```bash
# 1. 离线做好的 GPTQ / AWQ checkpoint：直接 serve，格式自动识别
vllm serve Qwen/Qwen2.5-7B-Instruct-AWQ --host 0.0.0.0 --port 8000

# 2. 在线量化：BF16 模型加载时压成 FP8，免离线步骤，适合快速试验
vllm serve Qwen/Qwen2.5-7B-Instruct --quantization fp8

# 3. GGUF：llama.cpp 生态的启动方式，同样暴露 OpenAI-compatible API
llama-server -m qwen2.5-7b-q4_k_m.gguf --port 8000
```

启动后回到本章第 3 节的健康检查，三种服务对客户端来说没有区别——量化只改变「模型怎么加载」，不改变 API 形状。比较 BF16 与量化版的 TTFT / TPOT / 吞吐差异，就是评测一本「最小对比」清单的第一次实战。

## 3. 健康检查与第一次请求

`vllm serve` 会占住 GPU 和终端，建议放在独立终端里跑；看到 `Uvicorn running on ...` 就是就绪信号。

客户端这边，第一步永远是健康检查——`GET /v1/models` 问一句「你在吗、加载了什么」。服务器在线就真的调一次对话；不在线就打印指引，去启动服务器再回来重跑。这个「先探测再使用」的模式本身也是部署的日常：

In [ ]:
import json
import urllib.request

BASE_URL = "http://localhost:8000/v1"
MODEL = "Qwen/Qwen3-0.6B"

def server_online(base_url=BASE_URL):
    """探测 OpenAI-compatible 服务器；在线返回 /v1/models 的解析结果，否则 None"""
    try:
        with urllib.request.urlopen(base_url + "/models", timeout=2) as r:
            return json.loads(r.read())
    except Exception:
        return None

def chat(prompt, max_tokens=128, temperature=0.7):
    """非流式调用 /v1/chat/completions，返回回复文本"""
    body = json.dumps({
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,   # 解码策略一本的参数，在这里真正生效
        "max_tokens": max_tokens,
    }).encode()
    req = urllib.request.Request(BASE_URL + "/chat/completions", data=body,
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=120) as r:
        return json.loads(r.read())["choices"][0]["message"]["content"]

info = server_online()
if info is None:
    print("服务器不在线。先在独立终端运行：")
    print("  vllm serve Qwen/Qwen3-0.6B --host 0.0.0.0 --port 8000")
    print("看到 Uvicorn running ... 后，重新执行本 cell")
else:
    print("服务器在线，已加载模型:", [m["id"] for m in info["data"]])
    print()
    print("回复:", chat("用一句话解释 KV Cache 是什么"))

In [ ]:
# 生态里更常见的写法：OpenAI 官方 SDK（pip install openai），协议完全一致
try:
    from openai import OpenAI

    client = OpenAI(base_url=BASE_URL, api_key="EMPTY")
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "用一句话解释 Prefill 和 Decode 的区别"}],
        temperature=0.7,
        max_tokens=128,
    )
    print(resp.choices[0].message.content)
except ImportError:
    print("未安装 openai 包（pip install openai）；上面的 urllib 版本不需要额外依赖")
except Exception as e:
    print("调用失败（多半是服务器未启动）：", type(e).__name__)

## 4. 流式输出

非流式请求要等全部生成完才返回；流式（`stream=True`）让服务端每生成几个 Token 就推一段（SSE）给客户端——这正是 ChatGPT「逐字往外蹦」的观感来源。

对这一章更有价值的是：流式响应的每个 chunk 都带时间戳，**TTFT 和 TPOT 的原始数据就在这里**。上一本这两个词还是纸面概念，现在要从一次真实请求里把它们量出来。

In [ ]:
import time

def stream_chat(prompt, max_tokens=64):
    """流式调用，返回 (每个 chunk 相对请求发出的时间戳, 拼接文本)"""
    body = json.dumps({
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "stream": True,
        "max_tokens": max_tokens,
    }).encode()
    req = urllib.request.Request(BASE_URL + "/chat/completions", data=body,
                                 headers={"Content-Type": "application/json"})
    stamps, text = [], ""
    t0 = time.time()
    with urllib.request.urlopen(req, timeout=120) as r:
        for raw in r:
            line = raw.decode().strip()
            if not line.startswith("data:"):
                continue
            payload = line[len("data:"):].strip()
            if payload == "[DONE]":
                break
            delta = json.loads(payload)["choices"][0].get("delta", {})
            piece = delta.get("content") or ""
            if piece:
                stamps.append(time.time() - t0)
                text += piece
    return stamps, text

if server_online():
    stamps, text = stream_chat("用两句话介绍 vLLM")
    ttft = stamps[0]
    tpot = (stamps[-1] - stamps[0]) / (len(stamps) - 1)
    print(text)
    print()
    print(f"TTFT = {ttft * 1000:.0f} ms, TPOT ≈ {tpot * 1000:.0f} ms（共 {len(stamps)} 个 chunk）")
else:
    print("服务器不在线：启动 vllm serve 后重跑本 cell")
    print("预期输出：一段逐字到达的文字 + TTFT / TPOT 两个毫秒数")

## 5. 测量 TTFT 与 TPOT

有了 chunk 时间戳，指标就是两行算术：TTFT 取第一个 chunk 的时刻，TPOT 取后续 chunk 的平均间距。先用一组演示时间戳把算法走清楚（不依赖服务器），再对在线服务做一次 8 并发小压测。

In [ ]:
def metrics_from_stamps(stamps):
    """从 chunk 时间戳（相对请求发出）算 (TTFT, TPOT)"""
    ttft = stamps[0]
    tpot = (stamps[-1] - stamps[0]) / (len(stamps) - 1)
    return ttft, tpot

demo = [0.42, 0.51, 0.60, 0.68, 0.77, 0.86, 0.94, 1.03]
ttft, tpot = metrics_from_stamps(demo)
print(f"TTFT = {ttft * 1000:.0f} ms   TPOT = {tpot * 1000:.0f} ms")
print()
print("关键观察：TTFT 看第一个 chunk，TPOT 看后续 chunk 的平均间距")
print("——正好对应 Prefill 和 Decode 两个阶段的体感")

In [ ]:
# 时间轴：先空等一段 TTFT，之后 Token 按稳定的节奏到达
import matplotlib.pyplot as plt

plt.figure(figsize=(6.5, 2.8))
plt.eventplot(demo, lineoffsets=1, linelengths=0.4, colors="tab:blue")
plt.axvspan(0, demo[0], color="tab:red", alpha=0.15)
plt.text(demo[0] / 2, 1.3, "TTFT", ha="center", color="tab:red")
plt.xlabel("time since request sent (s)")
plt.yticks([])
plt.title("Token arrival: wait TTFT, then stream at ~TPOT pace")
plt.show()

In [ ]:
# 8 并发小压测：把 TTFT / TPOT 从单请求扩展到分布
import concurrent.futures

def one_request(i):
    stamps, _ = stream_chat(f"用一句话解释概念 {i}：KV Cache")
    return metrics_from_stamps(stamps)

if server_online():
    with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
        results = list(pool.map(one_request, range(8)))
    ttfts = sorted(r[0] for r in results)
    tpots = sorted(r[1] for r in results)
    print(f"8 并发  TTFT: P50 {ttfts[4] * 1000:.0f} ms   最大 {ttfts[-1] * 1000:.0f} ms")
    print(f"        TPOT: P50 {tpots[4] * 1000:.0f} ms   最大 {tpots[-1] * 1000:.0f} ms")
    print()
    print("关键观察：改并发数、max_model_len 或换量化模型再跑一遍，")
    print("就是一次最小 serving 实验——指标变化对应推理系统一本里的每个机制")
else:
    print("服务器不在线；在线后重跑本 cell 可得到 8 并发的 TTFT / TPOT 分布")
    print("预期输出形如：8 并发  TTFT: P50 xxx ms   最大 xxx ms")

## 6. 用 SGLang 部署同一个模型

SGLang 的启动几乎一样简单，同样暴露 OpenAI-compatible API——所以前面几节的客户端代码原样复用：

```bash
pip install --upgrade pip uv
uv pip install --prerelease=allow sglang

python3 -m sglang.launch_server \
  --model-path Qwen/Qwen3-0.6B \
  --host 0.0.0.0 \
  --port 30000
```

生产环境也可以直接用官方 Docker 镜像。真正值得比较的不是谁的命令更短，而是上一本讲的那些机制各自的实现取舍：调度器行为、prefix cache（SGLang 的 RadixAttention 就在这里）、量化支持、多卡并行、chunked prefill。两边都跑通之后，「换引擎」就只是一次重启。

## 7. 启动参数与原理对照

部署时调的每个旋钮，都对应前几本里的一个机制。看到参数名应该能直接想起背后的问题：

| 参数 / 配置 | 背后的问题 | 出自 |
|:---|:---|:---|
| `--tensor-parallel-size` | 单卡放不下，切多卡 | 推理系统 · TP |
| `--max-model-len` | 最大上下文 = KV Cache 预算 | 推理为什么慢 · KV 账 |
| `--gpu-memory-utilization` | 留多少显存给 KV / runtime | 显存分配 |
| `--max-num-seqs` | 并发 active 序列上限 | 连续批的槽位数 |
| `--quantization` | 权重 / KV 低比特路径 | 量化 |
| prefix caching 开关 | 相同前缀不重算 | Prefix Cache |
| chunked prefill 配置 | 长 Prompt 调度 | Chunked Prefill |
| speculative config | 减少 Target decode 步数 | 投机解码 |

## 8. 常见问题排查

部署日常的另一半是排障。三类症状各有排查顺序，顺序本身就是前面章节的逻辑链：

```text
OOM
 -> 权重放不下？        -> 量化 / TP
 -> max_model_len 太大？ -> KV 预算超了 -> 调小
 -> 并发太高？          -> 降 max-num-seqs
 -> 还不够？            -> gpu-memory-utilization 再压

TTFT 高
 -> 排队？（吞吐饱和）   -> 扩容 / 调度
 -> Prompt 太长？       -> prefix cache 是否命中
 -> 被长请求阻塞？      -> chunked prefill 是否开启

TPOT 高
 -> Decode 吃带宽       -> 量化 / batch 是否够大
 -> batch 太大拖慢单请求 -> 降并发
 -> TP 通信开销？       -> 重新评估切分
```

每个「为什么」都能在前几本里找到答案——排障树就是课程内容的应用题。

## 9. 最小 Benchmark

部署验收最少记录这些，否则任何对比都没有意义：

```text
model / revision
hardware
dtype / quantization
max context / concurrency
input length / output length
TTFT P50 / P95
TPOT P50 / P95
throughput (tokens/s)
peak GPU memory
```

两边的基准工具都是现成的。

### 9.1 先用框架自己的 benchmark 工具跑基线

```bash
vllm bench serve --help
vllm bench throughput --help
vllm bench latency --help
```

SGLang 也有自己的 benchmark / profiling 工具链。第一次部署最重要的不是追某个漂亮数字，而是固定 workload——input length、output length、request rate、concurrency、硬件和量化配置全一致，两份报告才可比。这正是评测一本「公平比较」原则在性能侧的应用。

## 10. 部署视角的 PD 分离

单机 `vllm serve` 是一体化 serving。规模变大以后，系统可能进一步拆成：

```text
Gateway
   ↓
Prefill workers
   ↓  KV transfer
Decode workers
   ↓
Streaming response
```

这就是推理系统一本讲的 PD 分离落到部署拓扑后的样子。读厂商报告时先问四个问题：Prefill 和 Decode 是否分池？KV 怎么传？调度器在哪？目标是 TTFT、TPOT、吞吐还是成本？

在当前 vLLM 实现里还会看到 `KV connector`、NIXL、LMCache 这些词——它们处在 Prefill → Decode 的 KV 传输 / 远端缓存这一层，是实现名，不要和 PD 分离这个概念本身混为一谈。

## 11. 招聘 JD 解读

拿一条典型的 JD 检验这七本 notebook 的成果：

> 熟悉 vLLM / SGLang，理解 PagedAttention、Continuous Batching、Prefix Caching、Chunked Prefill、Speculative Decoding、PD Disaggregation；有量化和多卡推理经验。

现在这条链上的每个词都不再是名词，而是一段你实现过、模拟过、测量过的机制：

```text
Sampling（解码策略）
→ Prefill / Decode / KV Cache（推理为什么慢）
→ Quantization（量化）
→ Speculative Decoding（投机解码）
→ Scheduler / PagedAttention / Prefix Cache / Chunked Prefill / PD（推理系统）
→ 质量与性能的公平比较（评测）
→ vLLM / SGLang 部署与测量（本章）
```

这就是 Part 3 的终点：不仅「听过名词」，而是知道每个机制为什么出现、在系统哪一层、怎么亲手跑起来。

## 小结

- `vllm serve` 把 checkpoint 变成 OpenAI-compatible API；SGLang 同协议，客户端代码通用
- 客户端三件套：健康检查、普通调用、流式——chunk 时间戳就是 TTFT / TPOT 的原始数据
- 启动参数每个对应一个原理：TP ↔ 多卡、max_model_len ↔ KV 预算、quantization ↔ 低比特
- OOM / TTFT 高 / TPOT 高各有一棵排查树，顺序就是课程的应用题
- 公平比较两个引擎的前提：固定 workload（模型、精度、长度分布、并发、硬件全一致）
- 规模再往上就是 PD 分离：Prefill 池 + KV Transfer + Decode 池

到这里，Part 3 从「logits 怎么变成 Token」一路走到了「模型怎么变成服务」。

## 作业

三道题都是部署时的日常小活：读健康检查、算延迟指标、估并发容量。

> **关于 AI 辅助**：可以让 AI 提示思路、拆解步骤，但不建议直接让 AI 完成题目。

### 作业 1：解析 `/v1/models` 的响应

健康检查之后第一件事：看服务器到底加载了哪些模型。

**小提示**：响应结构是 `{"data": [{"id": ...}, ...]}`。

In [ ]:
# 作业 1：解析 /v1/models 响应 填空

import json

def list_models(raw_text):
    """从 /v1/models 的响应文本里抽出模型 id 列表"""
    # TODO：把下面三引号里的内容替换成你的代码
    """json.loads 之后取 data 里每个条目的 id"""

sample = '{"object":"list","data":[{"id":"Qwen/Qwen3-0.6B"},{"id":"bge-m3"}]}'
assert list_models(sample) == ["Qwen/Qwen3-0.6B", "bge-m3"]
print("✅ 作业 1 通过：健康检查的下一步永远是看服务器加载了什么模型")

### 作业 2：从 chunk 时间戳算 TTFT / TPOT

`stamps[i]` 是第 i 个 chunk 相对请求发出的秒数。TTFT 是第一个 chunk 的时间；
TPOT 是后续 chunk 的平均间距。

**小提示**：间距数比 chunk 数少一。

In [ ]:
# 作业 2：TTFT / TPOT 计算 填空

def metrics_from_stamps(stamps):
    """返回 (TTFT, TPOT)"""
    ttft = stamps[0]
    # TODO：把下面三引号里的内容替换成你的代码
    """tpot = (最后一个时间戳 - 第一个) / (chunk 数 - 1)"""
    return ttft, tpot

ttft, tpot = metrics_from_stamps([0.5, 0.6, 0.7, 0.8])
assert abs(ttft - 0.5) < 1e-9
assert abs(tpot - 0.1) < 1e-9
print("✅ 作业 2 通过：Streaming 体验的好坏就藏在这两个数里")

### 作业 3：估算还能开多少并发

权重和 KV Cache 都要装进显存：`可容纳请求数 = (总显存 - 权重) // 每请求 KV`。

**小提示**：用整除 `//`，答案向下取整。

In [ ]:
# 作业 3：并发容量估算 填空

def max_concurrency(vram_gb, weights_gb, kv_per_request_gb):
    """返回剩余显存还能容纳多少个请求的 KV Cache"""
    # TODO：把下面三引号里的内容替换成你的代码
    """(vram_gb - weights_gb) 整除 kv_per_request_gb"""

assert max_concurrency(24, 14, 1) == 10
assert max_concurrency(24, 14, 2) == 5
print("✅ 作业 3 通过：OOM 排查的第一步就是算这笔账（24GB 卡跑 7B BF16 只剩 10 个请求位）")